# Lesson 4.4: Sample Proportions

Simulate thousands of polls to see that the spread of the sample proportion matches the formula square root of p(1-p)/n.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. One poll

The lesson's poll: the true support for a bill is p = 0.54 and we ask n = 1,500 voters. Each voter says yes with probability p, so the number of yes answers is a binomial draw.

In [1]:
import numpy as np

rng = np.random.default_rng(2024)
p, n = 0.54, 1500

yes = rng.binomial(n, p)
print(f'yes answers: {yes} of {n}')
print(f'sample proportion p-hat = {yes / n:.4f}')

yes answers: 781 of 1500
sample proportion p-hat = 0.5207


## 2. Ten thousand polls

Now run the same poll 10,000 times. Each p-hat differs a little because of sampling variability. Their average should be near p, and their SD should be near the formula.

In [2]:
p_hats = rng.binomial(n, p, size=10_000) / n

print(f'mean of p-hats          = {p_hats.mean():.4f}')
print(f'SD of p-hats            = {p_hats.std(ddof=1):.4f}')
print(f'sqrt(p(1-p)/n) formula  = {np.sqrt(p * (1 - p) / n):.4f}')

mean of p-hats          = 0.5400
SD of p-hats            = 0.0130
sqrt(p(1-p)/n) formula  = 0.0129


## 3. Small samples wobble more

The lesson's three cases. The last column checks the rule that n times p and n times (1 minus p) should both be at least 10 for the normal model to fit well.

In [3]:
print('   n     p   SD of p-hats   formula   both n*p, n*(1-p) >= 10?')
for n, p in ((20, 0.35), (200, 0.40), (1500, 0.54)):
    ph = rng.binomial(n, p, size=10_000) / n
    ok = n * p >= 10 and n * (1 - p) >= 10
    print(f'{n:5d}  {p:.2f}   {ph.std(ddof=1):11.4f}   {np.sqrt(p * (1 - p) / n):7.4f}   {ok}')

   n     p   SD of p-hats   formula   both n*p, n*(1-p) >= 10?
   20  0.35        0.1053    0.1067   False
  200  0.40        0.0345    0.0346   True
 1500  0.54        0.0129    0.0129   True


## 4. The margin of error

A 95% margin of error is about 1.96 standard errors. If that is right, about 95% of polls should land within that distance of the true p.

In [4]:
n, p = 1500, 0.54
moe = 1.96 * np.sqrt(p * (1 - p) / n)
p_hats = rng.binomial(n, p, size=10_000) / n
inside = (np.abs(p_hats - p) <= moe).mean()

print(f'margin of error = {moe:.4f}')
print(f'share of polls within it = {inside:.3f}')

margin of error = 0.0252
share of polls within it = 0.951


## Try this

- Change p to 0.50 and predict whether the SD of the p-hats goes up or down.
- Change n to 6000 and predict the new SD (it should be half of the n = 1500 value).
- Try n = 20 with p = 0.02 and notice that many polls give exactly 0. Is the bell shape still good?

## The same idea in R, SPSS and Stata

**R**

```r
set.seed(2024)
p <- 0.54; n <- 1500
p_hats <- rbinom(10000, n, p) / n
mean(p_hats)
sd(p_hats)
sqrt(p * (1 - p) / n)
```

**SPSS** (Transform > Compute Variable (RV.BINOM), then Analyze > Descriptive Statistics > Descriptives)

```
SET SEED=2024.
INPUT PROGRAM.
LOOP poll = 1 TO 10000.
COMPUTE phat = RV.BINOM(1500, 0.54) / 1500.
END CASE.
END LOOP.
END FILE.
END INPUT PROGRAM.
DESCRIPTIVES VARIABLES=phat /STATISTICS=MEAN STDDEV.
```

**Stata**

```
clear
set seed 2024
set obs 10000
generate phat = rbinomial(1500, 0.54) / 1500
summarize phat
display sqrt(0.54 * 0.46 / 1500)
```